# 05 — Payload e qualidade da reconstrução

Executa ou lê a avaliação pareada de bytes, compressão, SNR e similaridade log-Mel para exatamente as mesmas janelas.

> Execute as células de cima para baixo. Parâmetros editáveis aparecem em uma célula própria e todo artefato gerado informa seu destino.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "data" / "freeze-spec-v1.json").is_file():
    ROOT = ROOT.parent
if not (ROOT / "data" / "freeze-spec-v1.json").is_file():
    raise RuntimeError("Execute este notebook dentro do repositório CodEco.")

for location in (ROOT / "src", ROOT / "scripts"):
    if str(location) not in sys.path:
        sys.path.insert(0, str(location))

REPORTS = ROOT / "reports"
DERIVED = ROOT / "data" / "derived"
REPORTS.mkdir(exist_ok=True)
DERIVED.mkdir(exist_ok=True)
print(f"Repositório: {ROOT}")


In [ ]:
RUN_FULL_EVALUATION = False  # altere somente no EASNCloud após validar o notebook 04


In [ ]:
import pandas as pd
import config as C

if RUN_FULL_EVALUATION:
    from run_codec_eval import main as run_evaluation
    run_evaluation()

results_path = C.OUT_DIR / "results.csv"
if results_path.is_file():
    results = pd.read_csv(results_path)
    display(results.head())
else:
    results = pd.DataFrame()
    print(f"Ainda não há resultados em {results_path.relative_to(ROOT)}")


In [ ]:
if not results.empty:
    summary = (results[results.codec != "baseline"]
               .groupby(["dataset", "codec", "setting"], as_index=False)
               .agg(bytes_mean=("nbytes", "mean"), bitrate_kbps=("bitrate_kbps", "mean"),
                    snr_db=("snr_db", "mean"), mel_rmse_db=("mel_rmse_db", "mean"),
                    mel_ssim=("mel_ssim", "mean")))
    summary.to_csv(REPORTS / "reconstruction_summary.csv", index=False)
    display(summary)


In [ ]:
if not results.empty:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 5))
    for (dataset, codec), group in summary.groupby(["dataset", "codec"]):
        ax.scatter(group.bitrate_kbps, group.mel_ssim, label=f"{dataset} — {codec}")
    ax.set_xscale("log"); ax.set(xlabel="bitrate efetivo (kbit/s)", ylabel="SSIM log-Mel")
    ax.legend(fontsize=7); fig.tight_layout(); plt.show()
